# Multi-Image Style Transfer Training on Kaggle

Use this notebook to train a style model from **multiple reference images** of the same style.
The trainer computes the mean Gram matrix across all N images, so the resulting model
captures a generalised version of the style rather than a single example.

**Before running any cell**, do these things in the Kaggle notebook UI:

1. **Add COCO 2017 dataset**  
   Notebook sidebar → *+ Add Data* → search **`awsaf49/coco-2017-dataset`** → Add.  
   This mounts the dataset at `/kaggle/input/datasets/awsaf49/coco-2017-dataset/` — no download needed.

2. **Enable GPU**  
   Notebook sidebar → *Settings* → Accelerator → **GPU T4 x1**.

3. **Upload your style images** into a folder under `/kaggle/working/`, e.g.  
   `/kaggle/working/my_style/` with `01.jpg`, `02.jpg`, etc.  
   Upload via: File browser (left panel) → create folder → Upload files.

Then run the cells below top-to-bottom.

## Step 1 — Verify GPU and COCO images are available

In [ ]:
import os
import pathlib

# ── GPU check ────────────────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ── COCO images check ────────────────────────────────────────────────────────
COCO_TRAIN = pathlib.Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017")
imgs = list(COCO_TRAIN.glob("*.jpg"))
print(f"COCO train images found: {len(imgs):,}")   # expect ~118,287
assert len(imgs) > 1000, "Dataset not mounted — add awsaf49/coco-2017-dataset in notebook settings"

## Step 1b — Verify internet access

**Internet must be enabled** for training to work — VGG16 weights (~550 MB) are downloaded on the first run.

Enable it in: *Notebook sidebar → Settings → Internet → On*  
Then re-run this cell. If the check passes, continue to Step 2.

In [ ]:
import urllib.request

try:
    urllib.request.urlopen("https://api.github.com", timeout=5)
    print("✓ Internet is available — safe to continue.")
except Exception:
    raise RuntimeError(
        "\n\n"
        "  ✗ No internet access detected.\n\n"
        "  Training requires internet to download VGG16 weights (~550 MB).\n"
        "  Fix: Notebook sidebar → Settings → Internet → On\n"
        "  Then re-run this cell before proceeding.\n"
    )

## Step 2 — Clone the repo and install trainer dependencies

In [ ]:
import subprocess, sys, os

REPO_URL = "https://github.com/PeterWazinski/style_transfer.git"
REPO_DIR = pathlib.Path("/kaggle/working/style_transfer")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)

# Install trainer deps (torch is already present on Kaggle GPU images)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[trainer]"], check=True)
print("Install done. Working dir:", os.getcwd())

## Step 3 — Configure style images directory

Upload your style images to a folder under `/kaggle/working/` via the Kaggle file browser  
(left panel → create folder → upload files).  
Supported formats: `.jpg`, `.jpeg`, `.png`.

Set `STYLE_IMAGES_DIR` to that folder path below.  
All images in the folder are automatically collected and used for training.

> ⚠ **Tip:** Use at least 3–5 images for meaningful style generalisation.  
>  A single image is valid but gives no multi-image benefit — use `kaggle_trainer.ipynb` instead.

**TV loss** (`TV_WEIGHT`) smooths out noise in the generated images. Default `1e-6` is a safe
starting value. Set to `0.0` to disable it entirely.

In [ ]:
import sys as _sys
_sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_training_helper import TrainingConfig, KaggleStyleRunner

# ── Edit these lines to match your job ───────────────────────────────────────
STYLE_IMAGES_DIR = pathlib.Path("/kaggle/working/sample_images/<style-dir>")  # ← folder with your style JPEGs
STYLE_ID         = "my_style"        # slug used as folder name and catalog ID
STYLE_NAME       = "My Style"        # display name shown in the gallery
TV_WEIGHT        = 1e-6              # Total Variation loss weight; set 0.0 to disable
# ─────────────────────────────────────────────────────────────────────────────

assert STYLE_IMAGES_DIR.is_dir(), (
    f"Style images directory not found: {STYLE_IMAGES_DIR}\n"
    "Create the folder and upload your style images via the Kaggle file browser."
)

cfg = TrainingConfig(
    style_images=[],               # will be populated by __post_init__ from style_images_dir
    style_images_dir=STYLE_IMAGES_DIR,
    style_id=STYLE_ID,
    style_name=STYLE_NAME,
    coco_path=COCO_TRAIN,
    style_weight=1e10,             # fixed: matches yakhyo reference (do not lower)
    content_weight=1e5,            # override if desired, e.g. 5e4 for more style influence
    tv_weight=TV_WEIGHT,
)
runner = KaggleStyleRunner(cfg)

print(f"Style images found: {len(cfg.style_images)}")
for i, p in enumerate(cfg.style_images, 1):
    print(f"  {i:>3}. {p.name}")

if len(cfg.style_images) == 0:
    raise ValueError(f"No .jpg/.jpeg/.png images found in {STYLE_IMAGES_DIR}")

## Step 3a — Per-image style analysis

Analyses texture complexity, edge density, and local variance for each style image.  
Images flagged with ⚠ are flat/low-contrast and may contribute less to the learned style.

In [ ]:
analysis = runner.analyse_style()

# ── Thumbnail strip ───────────────────────────────────────────────────────────
import io as _io
import ipywidgets as _widgets
from IPython.display import display as _display
from PIL import Image as _PILImage

def _thumb_widget(path: "pathlib.Path", size: int = 128) -> "_widgets.VBox":
    img = _PILImage.open(path).convert("RGB")
    img.thumbnail((size, size))
    buf = _io.BytesIO()
    img.save(buf, format="PNG")
    return _widgets.VBox([
        _widgets.Image(value=buf.getvalue(), format="png",
                       layout=_widgets.Layout(width=f"{size}px")),
        _widgets.Label(path.name[:18]),
    ])

thumbs = [_thumb_widget(p) for p in cfg.style_images[:12]]  # show up to 12
_display(_widgets.HBox(thumbs))

## Step 3b — Smoke test (~10 min on T4) ✅ run this before the 3-hour training

Trains for **2000 batches** using mean Gram matrices from all N style images,  
then measures how strongly the style is applied to a neutral content photo.

| Mean pixel change on content photo | Verdict | Action |
|---|---|---|
| < 8 | ⚠ Weak | Multiply `style_weight` × 10 in Step 3, re-run smoke test |
| 8 – 20 | ～ Moderate | Try `style_weight` × 3 for stronger effect |
| > 20 | ✓ Good | Proceed to Step 4 |

In [ ]:
do_smoke_test = True  # set to False to skip the smoke test and save time

if do_smoke_test:
    cfg.smoke_batches = 2000  # lower to 500 for a quicker check
    # cfg.content_weight = 5e4  # reduce for stronger style influence

    results = runner.run_smoke_test()

    # ── 3-way display: content | styled output | style reference ─────────────────
    import io as _io
    import ipywidgets as _widgets
    from IPython.display import display as _display

    def _to_widget(pil_img, width=210):
        buf = _io.BytesIO()
        pil_img.save(buf, format="PNG")
        return _widgets.Image(value=buf.getvalue(), format="png",
                            layout=_widgets.Layout(width=f"{width}px"))

    _w_content = _widgets.VBox([_widgets.Label("Content photo"),    _to_widget(results["content_img"])])
    _w_styled  = _widgets.VBox([_widgets.Label("Styled output"),    _to_widget(results["styled_img"])])
    _w_ref     = _widgets.VBox([_widgets.Label("Style ref (img 1)"), _to_widget(results["style_ref_img"])])
    _display(_widgets.HBox([_w_content, _w_styled, _w_ref]))
    print(f"Style images used in smoke test: {results['n_style_images']}")
else:
    print("Smoke test skipped. Set do_smoke_test=True to run it before full training.")

## Step 4 — Run training

Typical wall-clock times on a Kaggle T4:
| Epochs | COCO images | ~Time |
|---|---|---|
| 1 | 118 k | ~1.5 h |
| 2 | 118 k | ~3 h |

Adding N style images only adds seconds to precomputation — training time is unchanged.

Kaggle gives you **30 h/week** GPU, so 2 epochs comfortably fits in one session.

In [ ]:
# Launches full training (~3 h on T4 × 2 epochs = 236 k images).
# Saves config.json next to model.pth — resume_training() loads it automatically.
runner.run_full_training()

## Step 5 — Check preview and download the model

After training finishes you should have `model.onnx`, `model.pth`, and `preview.jpg` in the output directory.

Download everything via: Kaggle file browser → right-click on the output folder → *Download*.

Or zip and use the cell below to make a single download link.

In [ ]:
# Copies output files to /kaggle/output/<style_id>/ and creates a zip.
# Download from the Output tab (right panel) in the Kaggle notebook UI.
runner.package_output()

## Step 6 — Resume training from last checkpoint (if interrupted)

If training was interrupted, run this cell to continue from the last saved checkpoint.  
Checkpoint files are saved every 5 000 images as `model.ckpt_NNNNN.pth` inside the output folder.

The cell automatically picks the **most recent** checkpoint so you do not need to edit anything — just run it.

In [ ]:
# Resume training from the last checkpoint.
# Reads config.json automatically — no need to re-enter weights.
# cfg and runner must be configured (run Step 3 first if they are not).
runner.resume_training()